# Finding the Next Winner: Millionaire for Life

This notebook scrapes the latest draw history for Millionaire for Life and prepares it for analysis.


In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# Add the project root to sys.path to import the scraper
project_root = Path(os.getcwd()).parent.parent
sys.path.append(str(project_root))

from millionaire_life.scrape_millionaire_life import scrape_millionaire_life, write_csv, DEFAULT_OUTPUT

print(f"Project root: {project_root}")


## Scrape Latest Data


In [ ]:
# Scrape the data
rows = scrape_millionaire_life()

# Define output path relative to project root
# The scraper default is data/millionaire_life_history.csv relative to script
# We want millionaire_life/data/millionaire_life_history.csv relative to project root
output_path = project_root / "millionaire_life" / "data" / "millionaire_life_history.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Write to CSV
write_csv(rows, output_path)

print(f"Scraped {len(rows)} rows and saved to {output_path}")


## Load and Preview Data


In [ ]:
df = pd.read_csv(output_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date', ascending=False)
df.head()


## Train Model and Generate Predictions

This section runs `millionaire_life_mlx_ticket_model.py` to generate ranked ticket candidates.


In [ ]:
import subprocess

# Run the ticket model script
model_script = project_root / "millionaire_life" / "millionaire_life_mlx_ticket_model.py"
cmd = [
    sys.executable, str(model_script),
    "--csv", str(output_path),
    "--tickets", "10",
    "--ticket-type", "balanced",
    "--include-extra"
]

print(f"Running command: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(result.stdout)
else:
    print("Error running model:")
    print(result.stderr)


## Predict Next 5 Extra Numbers

This section runs `predict_5_extra.py` to forecast the next five 'Extra' numbers.


In [ ]:
# Run the extra numbers prediction script
predict_extra_script = project_root / "millionaire_life" / "predict_5_extra.py"
cmd_extra = [
    sys.executable, str(predict_extra_script),
    "--csv", str(output_path),
    "--epochs", "100"
]

print(f"Running command: {' '.join(cmd_extra)}")
result_extra = subprocess.run(cmd_extra, capture_output=True, text=True)

if result_extra.returncode == 0:
    print(result_extra.stdout)
else:
    print("Error running extra prediction:")
    print(result_extra.stderr)
